# LACT 观测数据重建

服务器版。数据和模型路径从 `PYLAST_OBSERVATION_CONFIG` 读取；模板见 `config/observation_reconstruction.example.json`。配置相对路径以该 JSON 的目录为准。
先读、校准和清洗一个实测事件，再查看单台与立体结果。当前模型仅在匹配的天顶角和布局域内试用。

In [ ]:
%matplotlib inline

import json, os, sys
from pathlib import Path
from IPython.display import display
from pylast.io import LactRealEventSource
from pylast.calib import LactRealCalibrator
from pylast.image import ImageProcessor
from pylast.visualize import (
    EventVisualizer, plot_raw_images, plot_clean_images,
    plot_event_cores, load_km2a_event_record, load_wcda_event_record,
)

cwd = Path.cwd()
repo = next(p for p in (cwd, *cwd.parents) if (p / "config/real_data.json").is_file())
sys.path.insert(0, str(repo / "notebooks"))
from observation_reconstruction import run_mono_demo, run_stereo_demo

config_path = Path(os.environ["PYLAST_OBSERVATION_CONFIG"]).expanduser().resolve()
run = json.loads(config_path.read_text(encoding="utf-8"))
def asset(name):
    return (config_path.parent / run[name]).resolve() if run.get(name) else None

config = json.loads((repo / "config/real_data.json").read_text(encoding="utf-8"))
source = LactRealEventSource(asset("input"), asset("mapping_file"),
    analysis_file=asset("analysis_file"), pointing_file=asset("pointing_file"),
    config_str=config["source"])
event = source[run["event_index"]]  # 索引从 0 开始
LactRealCalibrator(asset("calibration_file"), config_str=config["calibrator"])(event)
visualizer = EventVisualizer(source)
print(f"事件 {event.event_id} | LED={event.is_led} | 输入共有 {len(source)} 条")
_ = plot_raw_images(event, visualizer=visualizer)


## 清洗与联合图

KM2A / WCDA 路径留空即关闭；如配置，须与当前 LACT ROOT 属于同一观测文件。

In [ ]:
ImageProcessor(source.subarray, config_str=json.dumps(config["image_processor"]))(event)
external = {}
for name, key, loader in (("KM2A", "km2a_file", load_km2a_event_record),
                         ("WCDA", "wcda_file", load_wcda_event_record)):
    if asset(key) is not None:
        record = loader(asset(key), event.event_id)
        if record is not None:
            external[name] = record
if getattr(event.dl1, "tels", {}):
    _ = plot_clean_images(event, visualizer=visualizer, external_reconstructions=external)
if external:
    _ = plot_event_cores(event, visualizer=visualizer, image_level="dl0", external_reconstructions=external)


## 单台与立体模型

单台对每个可用真实 ID 分别推理。`fixed` 表示阈值来源，不代表把实测相机换成 tel7。
同一实测事件若只有一台，就没有立体解；可另配 `stereo_demo` 指向同布局、对应天顶角的模拟 DL1 ROOT，运行立体验证。

In [ ]:
run_mono = True
run_stereo = True
if run_mono:
    mono_results = run_mono_demo(event, source, config_path)
    if mono_results is not None:
        display(mono_results)
if run_stereo:
    stereo_results = run_stereo_demo(event, source, config_path)
    if stereo_results is not None:
        display(stereo_results)
